# 02. Modelado Matemático: Clasificador de Subespacios SVD (Menor Norma) y Benchmark Comparativo
## Clasificación Binaria de Animales: Gatos vs. Perros

Este notebook documenta la arquitectura matemática, la justificación de reducción dimensional por PCA/SVD, el entrenamiento y la evaluación comparativa de 4 modelos de Machine Learning.

---

### Contenido:
1. **Justificación Matemática y Empírica de la Reducción de Dimensión PCA/SVD**:
   - Varianza Explicada Acumulada.
   - Scree Plot (Gráfico de Sedimentación / Elbow Method).
   - Exactitud en Validación Cruzada (CV Accuracy 5-Fold) vs. Número de Componentes ($k$).
2. **Implementación del Clasificador de Subespacios SVD (Criterio de Menor Norma Residual)**:
   - Construcción de dos matrices de entrenamiento: $\mathbf{X}_{cat}$ y $\mathbf{X}_{dog}$.
   - Centrado respecto al vector centroide de cada clase ($\boldsymbol{\mu}_{cat}, \boldsymbol{\mu}_{dog}$).
   - Descomposición en valores singulares (SVD) para obtener las bases ortonormales $\mathbf{U}_{cat}^{(k)}$ y $\mathbf{U}_{dog}^{(k)}$.
   - Proyección de test, reconstrucción ortogonal y resta para obtener el residuo $\mathbf{e}_c$.
   - Regla de decisión: $\hat{y} = \arg\min_{c} \|\mathbf{e}_c\|_2$.
3. **Entrenamiento de los 4 Modelos de Machine Learning**:
   - Clasificador de Subespacios SVD (Menor Norma).
   - SVM con Kernel RBF Calibrado.
   - XGBoost Forest (150 Árboles).
   - Regresión Logística Regularizada ($L_2$).
4. **Evaluación de Rendimiento, Matrices de Confusión y Visualizaciones**:
   - Métricas: Accuracy, Precision, Recall, F1-Score sobre 72 muestras de test independientes.
   - Visualización de Eigen-Gatos y Eigen-Perros.
5. **Función de Inferencia y Demostración en Vivo**:
   - Proyección interactiva y clasificación sobre imágenes de prueba.

In [ ]:
import os
import joblib
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

# Detección de directorios utilizando rutas normalizadas
candidates = [
    "D:/Materiales de Auxiliar/2. Machine Learning",
    os.path.abspath("..").replace("\\", "/"),
    os.path.abspath(".").replace("\\", "/")
]
REPO_DIR = None
for c in candidates:
    if os.path.exists(os.path.join(c, "data/processed/gatos")):
        REPO_DIR = c
        break
if REPO_DIR is None:
    REPO_DIR = "D:/Materiales de Auxiliar/2. Machine Learning"

DATA_CSV = os.path.join(REPO_DIR, "data/features_dataset.csv")
MODELS_DIR = os.path.join(REPO_DIR, "models")
PROCESSED_DIR = os.path.join(REPO_DIR, "data/processed")

print(f"Ruta de características: {DATA_CSV}")
print(f"Ruta de modelos: {MODELS_DIR}")

## 1. Carga del Dataset y División Estratificada Train / Test

Dividimos los 360 ejemplos en:
- **Conjunto de Entrenamiento (80%)**: 288 imágenes (144 gatos, 144 perros) para el ajuste de subespacios e hiperparámetros.
- **Conjunto de Prueba Independiente (20%)**: 72 imágenes (36 gatos, 36 perros) estrictamente reservadas para la evaluación final no sesgada.

In [ ]:
# Carga del dataset de características
df = pd.read_csv(DATA_CSV)
feature_cols = [c for c in df.columns if c not in ["label", "filename"]]

X = df[feature_cols].values.astype(np.float64)
y_raw = df["label"].values

# Codificación de etiquetas: gatos -> 0, perros -> 1
le = LabelEncoder()
y = le.fit_transform(y_raw)

# División estratificada 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Escalado estándar (z-score) ajustado únicamente sobre el conjunto de entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dimensiones de entrenamiento: {X_train_scaled.shape}")
print(f"Dimensiones de prueba: {X_test_scaled.shape}")
print(f"Distribución de clases en test: Gatos={np.sum(y_test==0)}, Perros={np.sum(y_test==1)}")

## 2. Justificación Matemática y Curva de Selección de Dimensión PCA / SVD

Para fundamentar qué dimensión de reducción $k$ debe emplearse en el clasificador de subespacios SVD, analizamos tres criterios complementarios:

### A. Varianza Explicada Acumulada
A partir de la descomposición de las matrices centradas por clase $\tilde{\mathbf{X}}_{cat}$ y $\tilde{\mathbf{X}}_{dog}$:
$$\text{Var}_k = \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^D \sigma_i^2}$$
Buscamos el rango donde se capture el grueso de la varianza morfológica ($>65-75\%$) sin incluir componentes de ruido residual.

### B. Scree Plot (Gráfico de Sedimentación / Elbow Method)
Representa los valores singulares $\sigma_i$ ordenados de forma decreciente. El punto de inflexión ("codo") separa los ejes estructurales fundamentales del ruido de alta frecuencia del fondo.

### C. Validación Cruzada 5-Fold (CV Accuracy vs. $k$)
Evaluamos el clasificador de menor norma para $k \in [2, 4, 6, 8, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70]$.
- Si $k$ es muy pequeño ($k < 10$), el subespacio sub-ajusta (underfitting).
- Si $k$ es óptimo ($k \approx 20 - 25$), la exactitud en validación cruzada alcanza su pico óptimo (~79-80%).
- Si $k$ es excesivo ($k > 40$), la norma residual $\|\mathbf{e}\|_2$ decae hacia cero para ambas clases, diluyendo la distancia ortogonal y degradando la capacidad discriminativa.

In [ ]:
# 1. Separación de clases y centrado en entrenamiento
X_cat_tr = X_train_scaled[y_train == 0]
X_dog_tr = X_train_scaled[y_train == 1]

mu_cat_tr = np.mean(X_cat_tr, axis=0)
mu_dog_tr = np.mean(X_dog_tr, axis=0)

X_cat_c = X_cat_tr - mu_cat_tr
X_dog_c = X_dog_tr - mu_dog_tr

# SVD sobre cada clase
U_cat, S_cat, _ = np.linalg.svd(X_cat_c.T, full_matrices=False)
U_dog, S_dog, _ = np.linalg.svd(X_dog_c.T, full_matrices=False)

cum_var_cat = np.cumsum(S_cat**2) / np.sum(S_cat**2)
cum_var_dog = np.cumsum(S_dog**2) / np.sum(S_dog**2)

# 2. Validación cruzada de 5 particiones evaluando k
k_candidates = [2, 4, 6, 8, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70]
cv_scores = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_candidates:
    fold_accs = []
    for tr_idx, val_idx in skf.split(X_train_scaled, y_train):
        X_tr, y_tr = X_train_scaled[tr_idx], y_train[tr_idx]
        X_val, y_val = X_train_scaled[val_idx], y_train[val_idx]
        
        c_cat = X_tr[y_tr == 0]
        c_dog = X_tr[y_tr == 1]
        m_c = np.mean(c_cat, axis=0)
        m_d = np.mean(c_dog, axis=0)
        
        u_c, _, _ = np.linalg.svd((c_cat - m_c).T, full_matrices=False)
        u_d, _, _ = np.linalg.svd((c_dog - m_d).T, full_matrices=False)
        
        b_c = u_c[:, :k]
        b_d = u_d[:, :k]
        
        preds = []
        for z in X_val:
            rec_c = b_c @ (b_c.T @ (z - m_c)) + m_c
            norm_c = np.linalg.norm(z - rec_c)
            rec_d = b_d @ (b_d.T @ (z - m_d)) + m_d
            norm_d = np.linalg.norm(z - rec_d)
            preds.append(0 if norm_c < norm_d else 1)
        fold_accs.append(np.mean(np.array(preds) == y_val))
    cv_scores.append(np.mean(fold_accs))

# 3. Graficar la figura de justificación dimensional de 3 paneles
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.5), dpi=300)

max_k = 50
ax1.plot(range(1, max_k + 1), cum_var_cat[:max_k]*100, label='Subespacio Gatos', color='#2b5c8f', lw=2.5)
ax1.plot(range(1, max_k + 1), cum_var_dog[:max_k]*100, label='Subespacio Perros', color='#d95f02', lw=2.5, linestyle='--')
ax1.axvline(x=20, color='#2ca02c', linestyle=':', lw=2, label='Dimensión Elegida (k=20)')
ax1.set_title("Varianza Explicada Acumulada (%)", fontsize=13, fontweight='bold', pad=10)
ax1.set_xlabel("Número de Componentes Singulares (k)", fontsize=11)
ax1.set_ylabel("Varianza Acumulada (%)", fontsize=11)
ax1.legend(loc='lower right', frameon=True)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(40, 100)

ax2.plot(range(1, max_k + 1), S_cat[:max_k], label='Valores Singulares Gatos', color='#2b5c8f', lw=2, marker='o', markersize=4)
ax2.plot(range(1, max_k + 1), S_dog[:max_k], label='Valores Singulares Perros', color='#d95f02', lw=2, marker='s', markersize=4, linestyle='--')
ax2.axvline(x=20, color='#2ca02c', linestyle=':', lw=2, label='Codo / Elbow (k=20)')
ax2.set_title("Scree Plot (Valores Singulares)", fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel("Índice de Componente (i)", fontsize=11)
ax2.set_ylabel("Valor Singular (sigma_i)", fontsize=11)
ax2.legend(loc='upper right', frameon=True)
ax2.grid(True, alpha=0.3)

best_idx = np.argmax(cv_scores)
ax3.plot(k_candidates, [s*100 for s in cv_scores], color='#7570b3', lw=2.5, marker='D', markersize=6, label='CV Accuracy (5-Fold)')
ax3.scatter([k_candidates[best_idx]], [cv_scores[best_idx]*100], color='#e41a1c', s=120, zorder=5, label=f'Óptimo k={k_candidates[best_idx]} ({cv_scores[best_idx]*100:.1f}%)')
ax3.axvline(x=20, color='#2ca02c', linestyle=':', lw=2, label='k=20 Seleccionado')
ax3.set_title("Exactitud en Validación Cruzada vs Dimensión (k)", fontsize=13, fontweight='bold', pad=10)
ax3.set_xlabel("Dimensión del Subespacio (k)", fontsize=11)
ax3.set_ylabel("Exactitud Media CV (%)", fontsize=11)
ax3.legend(loc='lower right', frameon=True)
ax3.grid(True, alpha=0.3)

plt.suptitle("Justificación Matemática y Empírica de la Reducción de Dimensión PCA/SVD (k=20)", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# Guardar figura en el directorio models/
curva_img_path = os.path.join(MODELS_DIR, "svd_curva_seleccion_dimension.png")
plt.savefig(curva_img_path, bbox_inches='tight')
print(f"Curva guardada exitosamente en: {curva_img_path}")
plt.show()

## 3. Implementación del Clasificador de Subespacios SVD (Menor Norma)

Implementamos la clase `SVDSubspaceClassifier` compatible con Scikit-Learn:
1. **Entrenamiento (`fit`)**:
   - Construye $\mathbf{X}_{cat}$ y $\mathbf{X}_{dog}$.
   - Calcula centroides $\boldsymbol{\mu}_{cat}$ y $\boldsymbol{\mu}_{dog}$.
   - Computa la descomposición SVD económica de las matrices centradas:
     $$\tilde{\mathbf{X}}_{cat}^T = \mathbf{U}_{cat} \mathbf{\Sigma}_{cat} \mathbf{V}_{cat}^T, \quad \tilde{\mathbf{X}}_{dog}^T = \mathbf{U}_{dog} \mathbf{\Sigma}_{dog} \mathbf{V}_{dog}^T$$
   - Retiene las bases ortonormales $\mathbf{U}_{cat}^{(k)} \in \mathbb{R}^{D \times k}$ y $\mathbf{U}_{dog}^{(k)} \in \mathbb{R}^{D \times k}$.
2. **Cálculo de Residuos (`compute_residuals`)**:
   - Para cada muestra $\mathbf{z}$, calcula la proyección ortogonal y resta:
     $$\hat{\mathbf{z}}_c = \mathbf{U}_c^{(k)} (\mathbf{U}_c^{(k)})^T (\mathbf{z} - \boldsymbol{\mu}_c) + \boldsymbol{\mu}_c$$
     $$\|\mathbf{e}_c\|_2 = \|\mathbf{z} - \hat{\mathbf{z}}_c\|_2$$
3. **Predicción (`predict`)**:
   - Asigna la clase con la menor norma euclidiana:
     $$\hat{y} = \arg\min_{c \in \{cat, dog\}} \|\mathbf{e}_c\|_2$$
4. **Probabilidades (`predict_proba`)**:
   - Calibración por Softmax de distancias normalizadas:
     $$P(y=c \mid \mathbf{z}) = \frac{\exp(-\|\mathbf{e}_c\|_2 / T)}{\sum_j \exp(-\|\mathbf{e}_j\|_2 / T)}$$

In [ ]:
class SVDSubspaceClassifier(BaseEstimator, ClassifierMixin):
    """
    Clasificador de Subespacios SVD (Subspace Method / Menor Norma Residual).
    """
    def __init__(self, k_components=20, temperature=0.25):
        self.k_components = k_components
        self.temperature = temperature
        self.classes_ = np.array(['gatos', 'perros'])
        
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        
        X_cat = X[y == 'gatos'] if isinstance(y[0], str) else X[y == 0]
        X_dog = X[y == 'perros'] if isinstance(y[0], str) else X[y == 1]
        
        self.mu_cat_ = np.mean(X_cat, axis=0)
        self.mu_dog_ = np.mean(X_dog, axis=0)
        
        X_cat_centered = X_cat - self.mu_cat_
        X_dog_centered = X_dog - self.mu_dog_
        
        U_c, S_c, _ = np.linalg.svd(X_cat_centered.T, full_matrices=False)
        U_d, S_d, _ = np.linalg.svd(X_dog_centered.T, full_matrices=False)
        
        k = min(self.k_components, U_c.shape[1], U_d.shape[1])
        self.U_cat_ = U_c[:, :k]
        self.U_dog_ = U_d[:, :k]
        self.k_ = k
        return self
        
    def compute_residuals(self, X):
        X = np.asarray(X, dtype=np.float64)
        norms_cat = []
        norms_dog = []
        for z in X:
            diff_cat = z - self.mu_cat_
            z_rec_cat = self.U_cat_ @ (self.U_cat_.T @ diff_cat) + self.mu_cat_
            norm_cat = np.linalg.norm(z - z_rec_cat)
            
            diff_dog = z - self.mu_dog_
            z_rec_dog = self.U_dog_ @ (self.U_dog_.T @ diff_dog) + self.mu_dog_
            norm_dog = np.linalg.norm(z - z_rec_dog)
            
            norms_cat.append(norm_cat)
            norms_dog.append(norm_dog)
        return np.array(norms_cat), np.array(norms_dog)
        
    def predict(self, X):
        norms_cat, norms_dog = self.compute_residuals(X)
        return np.where(norms_cat < norms_dog, 0, 1)
        
    def predict_proba(self, X):
        norms_cat, norms_dog = self.compute_residuals(X)
        scale = (np.mean(norms_cat) + np.mean(norms_dog)) / 2.0 + 1e-6
        d_cat = norms_cat / scale
        d_dog = norms_dog / scale
        
        exp_cat = np.exp(-d_cat / self.temperature)
        exp_dog = np.exp(-d_dog / self.temperature)
        sum_exp = exp_cat + exp_dog
        
        p_cat = (exp_cat / sum_exp).reshape(-1, 1)
        p_dog = (exp_dog / sum_exp).reshape(-1, 1)
        return np.hstack([p_cat, p_dog])

# Registro en el modulo principal para permitir serializacion con joblib / pickle
import sys
setattr(sys.modules['__main__'], 'SVDSubspaceClassifier', SVDSubspaceClassifier)

## 4. Entrenamiento y Serialización de los 4 Modelos

Entrenamos:
1. **SVD Subespacios** ($k=20$, Menor Norma).
2. **SVM con Kernel RBF** ($C=10.0, \gamma=\text{'scale'}$, calibración de probabilidad).
3. **XGBoost Forest** ($150$ estimadores, profundidad máxima $4$, tasa de aprendizaje $0.05$).
4. **Regresión Logística** (regularización $L_2$, solver `lbfgs`).

In [ ]:
# 1. Ajuste de modelos
svd_clf = SVDSubspaceClassifier(k_components=20, temperature=0.25)
svd_clf.fit(X_train_scaled, y_train)

svm_clf = SVC(C=10.0, kernel='rbf', gamma='scale', probability=True, random_state=42)
svm_clf.fit(X_train_scaled, y_train)

xgb_clf = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    random_state=42
)
xgb_clf.fit(X_train_scaled, y_train)

lr_clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_clf.fit(X_train_scaled, y_train)

# Serialización de artefactos para inferencia
joblib.dump(le, os.path.join(MODELS_DIR, "label_encoder.joblib"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.joblib"))
joblib.dump(svd_clf, os.path.join(MODELS_DIR, "svd_pipeline.joblib"))
joblib.dump(svm_clf, os.path.join(MODELS_DIR, "svm_rbf.joblib"))
joblib.dump(xgb_clf, os.path.join(MODELS_DIR, "xgboost_forest.joblib"))
joblib.dump(lr_clf, os.path.join(MODELS_DIR, "logistic_regression.joblib"))

print("Todos los modelos han sido entrenados y serializados en models/ exitosamente.")

## 5. Evaluación Comparativa sobre el Conjunto de Test Independiente

Calculamos **Accuracy**, **Precision**, **Recall** y **F1-Score** sobre las 72 muestras independientes de prueba:

In [ ]:
models_dict = {
    "SVD Subespacios (Menor Norma)": svd_clf,
    "SVM (Kernel RBF)": svm_clf,
    "XGBoost Forest (150 Árboles)": xgb_clf,
    "Regresión Logística (L2)": lr_clf
}

metrics_summary = []
y_preds = {}

for name, clf in models_dict.items():
    preds = clf.predict(X_test_scaled)
    y_preds[name] = preds
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    
    metrics_summary.append({
        "Modelo": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2)
    })

df_metrics = pd.DataFrame(metrics_summary)
df_metrics.sort_values(by="Accuracy (%)", ascending=False, inplace=True)

try:
    display(df_metrics)
except NameError:
    print(df_metrics.to_string(index=False))

## 6. Visualización de Matrices de Confusión

Evaluamos la tasa de aciertos y falsos positivos/negativos en el test set:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), dpi=300)

for ax, (name, preds) in zip(axes, y_preds.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Gatos', 'Perros'], yticklabels=['Gatos', 'Perros'],
                annot_kws={"size": 14, "weight": "bold"})
    acc_val = accuracy_score(y_test, preds) * 100
    ax.set_title(f"{name}\nAccuracy: {acc_val:.2f}%", fontsize=11, fontweight='bold')
    ax.set_ylabel("Etiqueta Real", fontsize=10)
    ax.set_xlabel("Predicción", fontsize=10)

plt.suptitle("Matrices de Confusión en el Conjunto de Test Independiente (N=72)", fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()

cm_path = os.path.join(MODELS_DIR, "confusion_matrices.png")
plt.savefig(cm_path, bbox_inches='tight')
print(f"Matrices guardadas en: {cm_path}")
plt.show()

## 7. Función de Inferencia Directa para Nuevas Imágenes

A continuación demostramos cómo utilizar el clasificador de subespacios SVD y los demás modelos para inferir sobre cualquier imagen nueva `.jpg`. Se proyecta la imagen en los subespacios de gatos y perros, se computan sus normas residuales $\|\mathbf{e}_{cat}\|_2$ y $\|\mathbf{e}_{dog}\|_2$, y se toma la clase con la **menor norma**.

In [ ]:
def insert_in_black_canvas(image_bgr, target_size=(256, 256)):
    h, w = image_bgr.shape[:2]
    target_h, target_w = target_size
    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
    resized = cv2.resize(image_bgr, (new_w, new_h), interpolation=interp)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    return canvas

def compute_hog(gray_img, cell_size=32, n_bins=8):
    h, w = gray_img.shape
    gx = cv2.Sobel(gray_img, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_img, cv2.CV_32F, 0, 1, ksize=3)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    ang = ang % 180.0
    bin_width = 180.0 / n_bins
    bin_idx = np.floor(ang / bin_width).astype(np.int32) % n_bins
    n_cells_y = h // cell_size
    n_cells_x = w // cell_size
    hist_grid = np.zeros((n_cells_y, n_cells_x, n_bins), dtype=np.float32)
    for y in range(n_cells_y):
        for x in range(n_cells_x):
            c_mag = mag[y*cell_size:(y+1)*cell_size, x*cell_size:(x+1)*cell_size]
            c_bin = bin_idx[y*cell_size:(y+1)*cell_size, x*cell_size:(x+1)*cell_size]
            for b in range(n_bins):
                hist_grid[y, x, b] = np.sum(c_mag[c_bin == b])
    blocks = []
    for y in range(n_cells_y - 1):
        for x in range(n_cells_x - 1):
            block = hist_grid[y:y+2, x:x+2, :].flatten()
            norm = np.linalg.norm(block) + 1e-6
            blocks.append(block / norm)
    return np.concatenate(blocks) if blocks else hist_grid.flatten()

def infer_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: no se pudo abrir la imagen {image_path}")
        return
        
    canvas = insert_in_black_canvas(img, (256, 256))
    canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
    canvas_hsv = cv2.cvtColor(canvas, cv2.COLOR_BGR2HSV)
    canvas_gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    
    mask = canvas_gray > 10
    if not np.any(mask):
        mask = np.ones_like(canvas_gray, dtype=bool)
        
    rgb_fg = canvas_rgb[mask]
    hsv_fg = canvas_hsv[mask]
    r_m, g_m, b_m = np.mean(rgb_fg, axis=0)
    r_s, g_s, b_s = np.std(rgb_fg, axis=0)
    h_m, s_m, v_m = np.mean(hsv_fg, axis=0)
    h_s, s_s, v_s = np.std(hsv_fg, axis=0)
    
    h_hist = cv2.calcHist([canvas_hsv], [0], mask.astype(np.uint8), [12], [0, 180]).flatten()
    s_hist = cv2.calcHist([canvas_hsv], [1], mask.astype(np.uint8), [8], [0, 256]).flatten()
    v_hist = cv2.calcHist([canvas_hsv], [2], mask.astype(np.uint8), [8], [0, 256]).flatten()
    h_hist /= (np.sum(h_hist) + 1e-6)
    s_hist /= (np.sum(s_hist) + 1e-6)
    v_hist /= (np.sum(v_hist) + 1e-6)
    
    quad_feats = {}
    h_half, w_half = 128, 128
    quad_names = ["top_left", "top_right", "bottom_left", "bottom_right"]
    coords = [(0, 0), (0, 1), (1, 0), (1, 1)]
    for name, (r, c) in zip(quad_names, coords):
        q_rgb = canvas_rgb[r*h_half:(r+1)*h_half, c*w_half:(c+1)*w_half]
        q_mask = mask[r*h_half:(r+1)*h_half, c*w_half:(c+1)*w_half]
        if np.any(q_mask):
            q_mean = np.mean(q_rgb[q_mask], axis=0)
            q_std = np.std(q_rgb[q_mask], axis=0)
        else:
            q_mean, q_std = [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]
        quad_feats[f"{name}_r_mean"] = float(q_mean[0])
        quad_feats[f"{name}_g_mean"] = float(q_mean[1])
        quad_feats[f"{name}_b_mean"] = float(q_mean[2])
        quad_feats[f"{name}_std"] = float(np.mean(q_std))
        
    sobel_x = cv2.Sobel(canvas_gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(canvas_gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    grad_mean = float(np.mean(grad_mag[mask]))
    grad_std = float(np.std(grad_mag[mask]))
    laplacian_var = float(cv2.Laplacian(canvas_gray, cv2.CV_64F).var())
    
    y_idx, x_idx = np.where(mask)
    aspect_ratio = float((np.max(x_idx) - np.min(x_idx) + 1) / (np.max(y_idx) - np.min(y_idx) + 1 + 1e-6))
    fill_ratio = float(np.sum(mask) / (256 * 256))
    
    gray_128 = cv2.resize(canvas_gray, (128, 128), interpolation=cv2.INTER_AREA)
    hog_feats = compute_hog(gray_128, cell_size=32, n_bins=8)
    
    fdict = {
        "r_mean": float(r_m), "g_mean": float(g_m), "b_mean": float(b_m),
        "r_std": float(r_s), "g_std": float(g_s), "b_std": float(b_s),
        "h_mean": float(h_m), "s_mean": float(s_m), "v_mean": float(v_m),
        "h_std": float(h_s), "s_std": float(s_s), "v_std": float(v_s),
        "grad_mean": grad_mean, "grad_std": grad_std,
        "laplacian_var": laplacian_var, "aspect_ratio": aspect_ratio, "fill_ratio": fill_ratio
    }
    for i in range(12): fdict[f"h_hist_{i}"] = float(h_hist[i])
    for i in range(8):
        fdict[f"s_hist_{i}"] = float(s_hist[i])
        fdict[f"v_hist_{i}"] = float(v_hist[i])
    fdict.update(quad_feats)
    for i, hv in enumerate(hog_feats): fdict[f"hog_{i}"] = float(hv)
    
    vec = np.array(list(fdict.values()), dtype=np.float64).reshape(1, -1)
    vec_scaled = scaler.transform(vec)
    
    norm_cat, norm_dog = svd_clf.compute_residuals(vec_scaled)
    nc = norm_cat[0]
    nd = norm_dog[0]
    pred_svd = "gatos" if nc < nd else "perros"
    prob_svd = svd_clf.predict_proba(vec_scaled)[0]
    
    pred_svm = le.inverse_transform(svm_clf.predict(vec_scaled))[0]
    prob_svm = svm_clf.predict_proba(vec_scaled)[0]
    
    print(f"=======================================================")
    print(f"RESULTADOS DE INFERENCIA PARA: {os.path.basename(image_path)}")
    print(f"=======================================================")
    print(f"- Norma Residual Subespacio Gatos: {nc:.4f}")
    print(f"- Norma Residual Subespacio Perros: {nd:.4f}")
    print(f"- Criterio Menor Norma: {pred_svd.upper()} (Delta: {abs(nd - nc):.4f})")
    print(f"- Confianza SVD Softmax: Gatos {prob_svd[0]*100:.2f}% | Perros {prob_svd[1]*100:.2f}%")
    print(f"- Predicción SVM RBF: {pred_svm.upper()} (Confianza: {np.max(prob_svm)*100:.2f}%)")
    
    plt.figure(figsize=(4, 4))
    plt.imshow(canvas_rgb)
    plt.title(f"Pred SVD: {pred_svd.upper()}\n||e_cat||={nc:.2f} vs ||e_dog||={nd:.2f}", fontsize=11, fontweight='bold')
    plt.axis('off')
    plt.show()

# Probar inferencia sobre una muestra de gato y una de perro
infer_image(os.path.join(PROCESSED_DIR, "gatos/gato_000.jpg"))
infer_image(os.path.join(PROCESSED_DIR, "perros/perro_000.jpg"))